In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go

HERE = Path().resolve()
PROJECT_ROOT = next(p for p in [HERE, *HERE.parents] if (p / "src").exists())
sys.path.insert(0, str(PROJECT_ROOT)) 

from src.io.obd_loader import load_obd_csv, require_columns
from src.io.timebase import add_elapsed_time
from src.physics.kinematics import add_speed_ms, add_acceleration
from src.physics.longitudinal import VehicleParams, add_longitudinal_forces
from src.physics.power_energy import add_power_terms, add_energy_terms
from src.features.efficiency import add_chemical_efficiency


In [2]:
from dataHandler import clean_obd_csv
from dataSmoother import smooth_csv_file

In [3]:
cleaned_data_path = clean_obd_csv(PROJECT_ROOT/"data/KIT/2017-07-05_Seat_Leon_RT_S_Stau.csv")
preprocessed_df = load_obd_csv(cleaned_data_path)
preprocessed_df['time']
preprocessed_df.columns


Loaded: 2017-07-05_Seat_Leon_RT_S_Stau.csv
Rows: 46,349 | Cols: 11

Header standardisation:
  - 'Time' -> 'timestamp'
  - 'Engine Coolant Temperature [Â°C]' -> 'coolant_temp'
  - 'Intake Manifold Absolute Pressure [kPa]' -> 'map_kpa'
  - 'Engine RPM [RPM]' -> 'engine_rpm'
  - 'Vehicle Speed Sensor [km/h]' -> 'vehicle_speed'
  - 'Intake Air Temperature [Â°C]' -> 'intake_temp'
  - 'Air Flow Rate from Mass Flow Sensor [g/s]' -> 'maf_gps'
  - 'Absolute Throttle Position [%]' -> 'throttle_pct'
  - 'Ambient Air Temperature [Â°C]' -> 'ambient_temp'
  - 'Accelerator Pedal Position D [%]' -> 'pedal_pct_d'
  - 'Accelerator Pedal Position E [%]' -> 'pedal_pct_e'

Removed rows with blanks: 9
Remaining rows: 46,340

Saved cleaned CSV: C:\Users\alNM3\Desktop\DRIVESIM OFFICIAL CLONE LOCATION\DriveSim-AI-Lboro-Group\data\KIT\2017-07-05_Seat_Leon_RT_S_Stau_clean.csv



Index(['time', 'coolant_temp', 'map_kpa', 'engine_rpm', 'speed_kmh',
       'intake_temp', 'maf_gps', 'throttle_pct', 'ambient_temp', 'pedal_pct_d',
       'pedal_pct_e'],
      dtype='object')

In [4]:
# smoothed_data_path = smooth_csv_file(PROJECT_ROOT/"data/KIT/2017-07-05_Seat_Leon_RT_S_Stau.csv")
# cleaned_data_path = clean_obd_csv(smoothed_data_path)
# preprocessed_df = load_obd_csv(cleaned_data_path)
# preprocessed_df['time']
# preprocessed_df.columns

require_columns(preprocessed_df, ["time", "speed_kmh", "engine_rpm"])

preprocessed_df = add_elapsed_time(preprocessed_df)
preprocessed_df = add_speed_ms(preprocessed_df)
preprocessed_df = add_acceleration(preprocessed_df)

params = VehicleParams(
    mass_kg=1300, Cd=0.3, area_m2=2.2, crr=0.012,
    tyre_radius_m=0.318, rho_air=1.17
)

preprocessed_df = add_longitudinal_forces(preprocessed_df, params, grade_rad=0.0)
preprocessed_df = add_power_terms(preprocessed_df)
preprocessed_df = add_energy_terms(preprocessed_df)
preprocessed_df = add_chemical_efficiency(preprocessed_df)

KeyError: 'Air Flow Rate from Mass Flow Sensor [g/s]'

In [ ]:
# preprocessed_df["maf_gps"]
preprocessed_df.columns

In [ ]:
df = preprocessed_df.copy()



valid_speed = df["speed_ms"] > 5.0        
 
valid_drive = df["P_drive_W"] > 800  
valid_accel = df["accel_ms2"] > -0.3       

if "throttle_pct" in df.columns:
    valid_throttle = df["throttle_pct"] > 5
else:
    valid_throttle = True

mu = 0.8
F_max = mu * params.mass_kg * 9.81
valid_force = df["F_trac_N"].between(0, F_max)

valid_efficiency = df["chemical_efficiency"].between(0, 0.45)

valid = (valid_speed & valid_drive & valid_accel & valid_force & valid_efficiency & valid_throttle)

# Create cleaned efficiency column
df["chemical_efficiency_clean"] = df["chemical_efficiency"].where(valid)
df["chemical_efficiency_plot"] = df["chemical_efficiency_clean"].interpolate(limit=5)

In [ ]:
#df = df.iloc[10000:40000].copy()
#sub = df.iloc[20000:23000].copy()
#sub = df.iloc[0:].copy()
sub = df.iloc[21000:23000].copy()

plt.figure(figsize=(12, 5))

plt.plot(
    sub["elapsed_time_s"],
    sub["chemical_efficiency"],
    label="Raw efficiency",
    alpha=0.25,
    linewidth=1
)

plt.plot(
    sub["elapsed_time_s"],
    sub["chemical_efficiency_plot"],
    label="Physics-filtered efficiency",
    linewidth=2
)

plt.axhline(0.35, linestyle="--", label="Typical upper realistic range")
plt.axhline(0.45, linestyle="--", label="Hard limit")

plt.xlabel("Elapsed time [s]")
plt.ylabel("chemical efficiency: Pdrive / Pfuel")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# plt.figure(figsize=(12, 5))
# plt.plot(
#     sub["elapsed_time_s"],
#     sub["chemical_efficiency_plot"],
#     label="Physics-filtered efficiency",
#     linewidth=2
# )
# plt.xlabel("Elapsed time [s]")
# plt.ylabel("Pdrive / Pfuel")
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.show()

In [ ]:
# fig = go.Figure()

# fig.add_trace(go.Scatter(
#     x=sub["elapsed_time_s"],
#     y=sub["chemical_efficiency_plot"],
#     mode="lines",
#     name="Physics-filtered efficiency",
#     line=dict(width=2)
# ))

# fig.update_layout(
#     paper_bgcolor="white",
#     plot_bgcolor="white",
#     xaxis_title="Elapsed time [s]",
#     yaxis_title="Pdrive / Pfuel",
#     legend=dict(x=0, y=1),
#     width=1200,
#     height=500
# )
# fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
# fig.update_yaxes(showgrid=True, gridcolor="lightgrey")

# fig.show()

In [ ]:
mean_value = df["chemical_efficiency_clean"].mean()
print(mean_value)

## rolling energy

In [ ]:
dt = df["elapsed_time_s"].diff().fillna(0)
dt = dt.clip(lower=0, upper=2)

df["E_drive_step_J"] = df["P_drive_W"] * dt
df["E_fuel_step_J"] = df["Pfuel"] * dt

window = 1000
min_period = 100

df["eff_rolling_energy"] = (
    df["E_drive_step_J"].where(valid).rolling(window, min_periods=min_period).sum()
    /
    df["E_fuel_step_J"].where(valid).rolling(window, min_periods=min_period).sum()
)

df["eff_rolling_energy"] = df["eff_rolling_energy"].where(
    df["eff_rolling_energy"].between(0, 0.45)
)


#sub = df.iloc[15000:30000].copy()
#sub = df.iloc[0:].copy()
sub = df.iloc[21000:23000].copy()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    sub["elapsed_time_s"],
    sub["eff_rolling_energy"],
    label="Rolling energy efficiency",
    linewidth=2
)

#plt.axhline(0.20, linestyle="--", label="Typical lower-good range")
#plt.axhline(0.30, linestyle="--", label="Typical upper-good range")
#plt.axhline(0.45, linestyle=":", label="Hard sanity limit")

plt.xlabel("Elapsed time [s]")
plt.ylabel("Rolling efficiency: Edrive / Efuel")
plt.title("Rolling Fuel Efficiency")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
E_drive_total = df["E_drive_step_J"].sum()
E_fuel_total = df["E_fuel_step_J"].sum()

trip_efficiency = E_drive_total / E_fuel_total

In [ ]:
trip_efficiency

In [ ]:
# #filtering las min
# valid_speed = df["speed_ms"] > 5.0   # about 18 km/h
# valid_fuel = df["P_fuel_W"] > 5000 
# valid_drive = df["P_drive_W"] > 1000 
# valid_throttle = df["throttle_pct"] > 5
# 
# valid_accel = df["accel_ms2"] > -0.3

# valid_force = df["F_tr_Wheel_N"].between(0, mass_kg*0.8*9.81)

In [ ]:
# sub = preprocessed_df.iloc[:30000].copy()
# sub = preprocessed_df.iloc[0:].copy()

# plt.figure(figsize=(10,5))
# plt.plot(sub["elapsed_time_s"], sub["chemical_efficiency"], label="efficiency", linewidth=2)

# plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
#plt.figure(figsize=(10,5))
#plt.plot(sub["elapsed_time_s"], sub["F_trac_N"], label="efficiency", linewidth=2)

In [ ]:
# plt.figure(figsize=(10,5))
# plt.plot(sub["elapsed_time_s"], sub["P_drive_W"], label="efficiency", linewidth=2)

In [ ]:
# plt.figure(figsize=(10,5))
# plt.plot(sub["elapsed_time_s"], sub["Pfuel"], label="efficiency", linewidth=2)

In [ ]:
# plt.figure(figsize=(10,5))
# plt.plot(sub["elapsed_time_s"], sub["Pfuel"], sub["P_drive_W"], label="efficiency", linewidth=2)

In [ ]:
# result = (df['P_drive_W'] > df['Pfuel']).any()

# print(result) 

In [ ]:
# rows_where_true = df[df['P_drive_W'] > df['Pfuel']]
# print(rows_where_true)

In [ ]:
# plt.figure(figsize=(10,5))
# plt.plot(df["elapsed_time_s"], df["chemical_efficiency"], label="efficiency", linewidth=2)

# plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
# rows_where_false = df[df['P_drive_W'] < df['Pfuel']]

In [ ]:
# #sub = df.iloc[:30000].copy()

# plt.figure(figsize=(10,5))
# plt.plot(rows_where_false["elapsed_time_s"], rows_where_false["chemical_efficiency"], label="efficiency", linewidth=2)

# plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()